In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import pandas as pd
from pathlib import Path
import json
import time


load_dotenv(
    "/Users/yurujia/Desktop/Dissertation Data/sentiment/.env"
)

client = OpenAI()

MODEL = "gpt-5-mini"

In [2]:
def build_prompt_p2r(text):

    return f"""
You are performing sentiment classification for an academic research project on news coverage of autonomous vehicles.

Your task is to classify the overall evaluative orientation of the following news text toward autonomous vehicles, autonomous driving technology, or their development and deployment.

Use the following labels:

1 = Positive

Classify the text as positive when it predominantly presents autonomous vehicles or autonomous driving in a favorable direction.

Positive sentiment may be expressed explicitly or implicitly. Explicit positive words are not required.

Indicators may include:
- technological progress or successful development;
- successful testing, deployment, commercialization, or expansion;
- demonstrated improvements in safety or performance;
- benefits for users, society, mobility, or industry;
- supportive developments that facilitate autonomous driving;
- optimistic expectations about future development.

A factual description of successful progress or beneficial development may therefore still be classified as positive when its overall implication toward autonomous driving is clearly favorable.


0 = Neutral

Classify the text as neutral when it primarily provides information about autonomous vehicles without a discernible positive or negative evaluative direction.

Neutral should be used when:
- the text mainly reports facts without implying clear progress, benefit, risk, failure, or setback;
- positive and negative elements are both present and neither clearly dominates;
- autonomous driving is mentioned only as contextual information.

Do not classify a text as neutral merely because it is written in an objective news style.


-1 = Negative

Classify the text as negative when it predominantly presents autonomous vehicles or autonomous driving in an unfavorable direction.

Negative indicators include:
- technological failure;
- safety problems;
- crashes linked to autonomous systems;
- suspension or setbacks;
- criticism;
- regulatory problems;
- limitations or unreliability.

Important:

Evaluate sentiment specifically toward autonomous vehicles, autonomous driving technology, or their development and deployment.

Do not classify based on:
- company stock performance;
- executives;
- unrelated business issues.

When both positive and negative elements appear, classify according to the dominant evaluative direction.

Base the classification only on the supplied text.

Return only:

1
0
or
-1


News text:

{text}
""".strip()

In [4]:
USA_PATH = Path(
"/Users/yurujia/Desktop/Dissertation Data/USA/descriptive_stats_USA_Today/USA_Today_AV_final_with_first_av_relevant_paragraph.xlsx"
)


usa = pd.read_excel(USA_PATH)


usa_sentiment = pd.DataFrame({

    "id": range(len(usa)),
    "source": "USA Today",
    "text": usa["first_av_relevant_paragraph"]

})


print(usa_sentiment.shape)

(287, 3)


In [5]:
WSJ_PATH = Path(
"/Users/yurujia/Desktop/Dissertation Data/USA/descriptive_stats_WSJ/WSJ_AV_final_analytical_dataset.xlsx"
)


wsj = pd.read_excel(WSJ_PATH)


wsj_sentiment = pd.DataFrame({

    "id": range(len(wsj)),
    "source": "WSJ",
    "text": wsj["abstract"]

})


print(wsj_sentiment.shape)

(367, 3)


In [6]:
all_articles = pd.concat(
    [
        usa_sentiment,
        wsj_sentiment
    ],
    ignore_index=True
)


print(all_articles.shape)

(654, 3)


In [7]:
batch_file = Path(
    "/Users/yurujia/Desktop/Dissertation Data/USA/"
    "english_sentiment_batch_input.jsonl"
)


with open(batch_file,"w") as f:

    for idx,row in all_articles.iterrows():

        request = {

            "custom_id": f"article_{idx}",

            "method":"POST",

            "url":"/v1/responses",

            "body":{

                "model":MODEL,

                "input":
                    build_prompt_p2r(
                        row["text"]
                    )

            }

        }

        f.write(
            json.dumps(request)
            + "\n"
        )


print(
    "Batch file created:",
    batch_file
)

Batch file created: /Users/yurujia/Desktop/Dissertation Data/USA/english_sentiment_batch_input.jsonl


In [8]:
batch_input = client.files.create(
    file=open(
        batch_file,
        "rb"
    ),
    purpose="batch"
)


print(batch_input.id)

file-HCZc5s6iKQMkUxYhbqSMsr


In [9]:
batch_job = client.batches.create(

    input_file_id=batch_input.id,

    endpoint="/v1/responses",

    completion_window="24h"

)


print(batch_job.id)

batch_6a64d01c43408190bfaa30681826650b


In [13]:
status = client.batches.retrieve(
    batch_job.id
)


print(status.status)

completed


In [14]:
output_file_id = status.output_file_id


file_response = client.files.content(
    output_file_id
)


result_path = Path(
"/Users/yurujia/Desktop/"
"english_sentiment_batch_output.jsonl"
)


with open(
    result_path,
    "w"
) as f:

    f.write(
        file_response.text
    )


print(result_path)

/Users/yurujia/Desktop/english_sentiment_batch_output.jsonl


In [18]:
import json

with open(
    result_path,
    encoding="utf-8"
) as f:

    first_line = json.loads(
        f.readline()
    )


first_line.keys()

dict_keys(['id', 'custom_id', 'response', 'error'])

In [19]:
first_line["response"].keys()

dict_keys(['status_code', 'request_id', 'body'])

In [20]:
first_line["response"]["body"].keys()

dict_keys(['id', 'object', 'created_at', 'status', 'background', 'billing', 'completed_at', 'error', 'frequency_penalty', 'incomplete_details', 'instructions', 'max_output_tokens', 'max_tool_calls', 'model', 'moderation', 'output', 'parallel_tool_calls', 'presence_penalty', 'previous_response_id', 'prompt_cache_key', 'prompt_cache_retention', 'reasoning', 'safety_identifier', 'service_tier', 'store', 'temperature', 'text', 'tool_choice', 'tool_usage', 'tools', 'top_logprobs', 'top_p', 'truncation', 'usage', 'user', 'metadata'])

In [24]:
import json

with open(
    result_path,
    encoding="utf-8"
) as f:

    for line in f:
        item = json.loads(line)

        if item["response"]["status_code"] == 200:

            body = item["response"]["body"]

            print(body.keys())

            print("\nOUTPUT:")
            print(body.get("output"))

            break

dict_keys(['id', 'object', 'created_at', 'status', 'background', 'billing', 'completed_at', 'error', 'frequency_penalty', 'incomplete_details', 'instructions', 'max_output_tokens', 'max_tool_calls', 'model', 'moderation', 'output', 'parallel_tool_calls', 'presence_penalty', 'previous_response_id', 'prompt_cache_key', 'prompt_cache_retention', 'reasoning', 'safety_identifier', 'service_tier', 'store', 'temperature', 'text', 'tool_choice', 'tool_usage', 'tools', 'top_logprobs', 'top_p', 'truncation', 'usage', 'user', 'metadata'])

OUTPUT:
[{'id': 'rs_08ae2c215fa8e0c6006a64d0b87ff08195bfbf437e7bfa3fbf', 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqZNC5sGiQhD0gk2p3GAFmySruoj8Ss8rRWkyl2j0Y0SZV11eFDlSh-dIquREAPwPi7wYgTkeVzsZI2OkE39zImTlFH7qC1OjMNN1Xa16EbxUmT3P86KVqzkjbjO2Gkatk6BNaqm48YvqA4eic7Uhg1BmH7M2UkXk8-9Qsr50jo9iKf9gz-hM_tIxdclL95LtnAF23r9kcOhzl6DuaT06XBfaakw3B8_olwdbOKHcicpFMQUW68gJkN0V4ZbBMv4QlgZzVM3c3_NyJxz-Qy4KJxct1xsbO5xiIksNfzm-IBqp4Dilk_Ovm4mtmeWYtrtHPYY8YS1

In [25]:
# %%
# ============================================================
# Parse Batch output correctly
# ============================================================

predictions = []


with open(
    result_path,
    encoding="utf-8"
) as f:


    for line in f:

        item = json.loads(line)


        article_id = int(
            item["custom_id"]
            .replace(
                "article_",
                ""
            )
        )


        response_body = (
            item["response"]["body"]
        )


        output_text = None


        # Find assistant message output
        for output_item in response_body["output"]:


            if output_item.get("type") == "message":


                for content_item in output_item.get(
                    "content",
                    []
                ):


                    if (
                        content_item.get("type")
                        ==
                        "output_text"
                    ):

                        output_text = (
                            content_item["text"]
                        )


        if output_text is None:

            print(
                "No output found:",
                article_id
            )

            continue


        output_text = (
            output_text
            .strip()
        )


        if output_text in [
            "1",
            "0",
            "-1"
        ]:


            predictions.append({

                "id":
                    article_id,

                "sentiment":
                    int(output_text)

            })


        else:

            print(
                "Unexpected output:",
                article_id,
                repr(output_text)
            )


prediction_df = pd.DataFrame(
    predictions
)


print(
    "Parsed predictions:",
    len(prediction_df)
)


display(
    prediction_df.head()
)

Parsed predictions: 654


,id,sentiment
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1


In [26]:
final_sentiment_results = (
    all_articles
    .merge(
        prediction_df,
        on="id"
    )
)


final_sentiment_results[
    "model"
]="gpt-5-mini"


final_sentiment_results[
    "prompt"
]="P2R-EN"


final_sentiment_results[
    "date_processed"
]=pd.Timestamp.now()


display(
    final_sentiment_results.head()
)

,id,source,text,sentiment,model,prompt,date_processed
0,0,USA Today,"Fiat Chrysler Automobiles revealed a new, semi...",1,gpt-5-mini,P2R-EN,2026-07-25 23:14:13.269303
1,1,USA Today,Fiat Chrysler and Google already have a partne...,1,gpt-5-mini,P2R-EN,2026-07-25 23:14:13.269303
2,2,USA Today,The technology company confirmed that it's tak...,1,gpt-5-mini,P2R-EN,2026-07-25 23:14:13.269303
3,3,USA Today,"Tesla Motors CEO Elon Musk is letting 1,000 cu...",1,gpt-5-mini,P2R-EN,2026-07-25 23:14:13.269303
4,4,USA Today,"Nvidia is at the intersection of key, changing...",1,gpt-5-mini,P2R-EN,2026-07-25 23:14:13.269303


In [27]:
OUTPUT = Path(
"/Users/yurujia/Desktop/"
"english_AV_sentiment_final_P2R_EN.xlsx"
)


final_sentiment_results.to_excel(
    OUTPUT,
    index=False
)


print(OUTPUT)

/Users/yurujia/Desktop/english_AV_sentiment_final_P2R_EN.xlsx
